In [2]:
import backtrader as bt

In [ ]:
def run_complete_backtest(strategy_class, symbol, start_date, end_date, **kwargs):
    """
    完整的回测运行框架
    """
    cerebro = bt.Cerebro()
    
    # 添加策略
    cerebro.addstrategy(strategy_class, **kwargs)
    
    # 获取数据
    data = bt.feeds.PandasData(
        dataname=yf.download(symbol, start_date, end_date)
    )
    cerebro.adddata(data)
    
    #  broker设置
    cerebro.broker.setcash(100000.0)
    cerebro.broker.setcommission(commission=0.001)
    
    # 添加分析器
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    cerebro.addanalyzer(bt.analyzers.AnnualReturn, _name='annual')
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')
    cerebro.addanalyzer(bt.analyzers.TimeReturn, _name='timereturn')
    
    # 运行回测
    print(f'=== {strategy_class.__name__} 回测结果 ===')
    print('初始资金: %.2f' % cerebro.broker.getvalue())
    
    results = cerebro.run()
    strat = results[0]
    
    final_value = cerebro.broker.getvalue()
    print('最终资金: %.2f' % final_value)
    
    # 性能分析
    sharpe = strat.analyzers.sharpe.get_analysis()
    drawdown = strat.analyzers.drawdown.get_analysis()
    trades = strat.analyzers.trades.get_analysis()
    
    print(f'夏普比率: {sharpe.get("sharperatio", 0):.2f}')
    print(f'最大回撤: {drawdown.get("max", {}).get("drawdown", 0):.2f}%')
    print(f'总交易次数: {trades.get("total", {}).get("total", 0)}')
    print(f'盈利交易比例: {trades.get("won", {}).get("total", 0)/max(trades.get("total", {}).get("total", 1), 1)*100:.1f}%')
    
    # 绘图
    cerebro.plot()
    
    return strat

# 运行所有策略
if __name__ == '__main__':
    # 安装所需库: pip install backtrader yfinance pandas
    
    # 测试双均线策略
    run_complete_backtest(
        DoubleMASStrategy, 'AAPL', '2020-01-01', '2023-12-31',
        fast=10, slow=30
    )
    
    # 测试RSI策略
    run_complete_backtest(
        RSIStrategy, 'MSFT', '2020-01-01', '2023-12-31',
        rsi_period=14, rsi_oversold=30, rsi_overbought=70
    )